# SQL analysis of merchandise purchases

## Summary

Saved SQL queries reconcile with pandas across every returned row and measure. The reviewed population contains **19,773 eligible purchase invoices**, gross merchandise purchase value of **GBP 10.27 million**, and gross average order value of **GBP 519.47**. The United Kingdom accounts for **85.16%** of gross purchase value. Among identified customers, **65.27%** have at least two eligible purchase invoices in the extract.

All values are **before credits** and retain exact copies. Product 23843 ranks second by value from a single 80,995-unit purchase line, illustrating the sensitivity of gross rankings to extreme transactions. December 2011 is partial; no full-month growth comparison is calculated for it.

## Context and methods

Source: Chen, D. (2015), *Online Retail*, UCI Machine Learning Repository, DOI [10.24432/C5BW33](https://doi.org/10.24432/C5BW33), CC BY 4.0. Coverage: 1 December 2010 to 9 December 2011. Currency: GBP. Source timezone unspecified.

### Key assumptions

- Use the verified Stage 2 purchase populations; preserve anonymous purchases in sales and restrict customer calculations to known IDs.
- Keep all source rows in a separate analytical ledger; raw SQLite and the source workbook remain unchanged.
- Count distinct eligible invoices as orders. Group each invoice to its earliest eligible timestamp for order-month attribution; retain all eligible line value in that order month.
- Retain exact copies and numeric extremes; no credit matching or net revenue is assumed.
- Aggregate products by normalized stock code. Use one latest eligible description per code as a display label, avoiding joins that multiply values.
- Sum integer thousandths of GBP. Float conversion is for ratios/display only.
- Customer repeat purchasing means at least two distinct eligible invoices in this historical window; it is not a retention or lifetime-value measure.

Methods and query references: [SQL analysis](../docs/sql_analysis.md). Charts, deeper comparisons, and recommendations belong to the subsequent EDA and insights stages; this notebook records the tabular SQL baseline.

## Data

### Verify analytical ingestion and execute saved SQL

The loader validates current workbook/classification/export fingerprints and the existing QA receipt, reconciles every SQL ledger field, and checks invoice dimensions. A repeat run verifies the existing analytical database without appending records. The runner independently reconciles complete query outputs against pandas.

In [1]:
from pathlib import Path
from decimal import Decimal
import sys
import json
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT / "scripts"))
import run_sql_analysis
report=run_sql_analysis.main()
print("Six saved analytical SQL queries reconciled in full.")

existing_database_verified: 541,909 ledger rows; every field and all purchase populations reconciled.


{
  "reconciliation": {
    "sales_summary": "all_rows_and_measures_passed",
    "products": "all_rows_and_measures_passed",
    "countries": "all_rows_and_measures_passed",
    "monthly_sales": "all_rows_and_measures_passed",
    "orders": "all_rows_and_measures_passed",
    "customers": "all_rows_and_measures_passed"
  },
  "sales_summary": {
    "purchase_lines": 527761,
    "purchase_orders": 19773,
    "identified_customers": 4334,
    "units": 5577103,
    "gross_purchase_millipounds": 10271433063,
    "identified_purchase_millipounds": 8761066653,
    "anonymous_purchase_millipounds": 1510366410,
    "gross_aov_gbp": 519.4676105295099
  },
  "customer_purchase_summary": {
    "identified_customers": 4334,
    "one_order_customers": 1505,
    "repeat_customers": 2829,
    "repeat_customer_share_percent": 65.27457314259345
  }
}
Six saved analytical SQL queries reconciled in full.


## Results

### Sales value and order count

Gross average order value divides total eligible merchandise value by distinct eligible purchase invoices. It does not divide by line count. These are purchase values before credits, not net revenue or profit.

In [2]:
def gbp(millipounds):
    return f"{Decimal(int(millipounds))/Decimal(1000):,.3f}"

s=report["sales_summary"]
display(pd.DataFrame([
    ("Eligible purchase lines", f"{s['purchase_lines']:,}"),
    ("Eligible purchase invoices", f"{s['purchase_orders']:,}"),
    ("Identified customers", f"{s['identified_customers']:,}"),
    ("Gross purchase value (GBP)",gbp(s["gross_purchase_millipounds"])),
    ("Gross average order value (GBP)",f"{s['gross_aov_gbp']:,.2f}"),
],columns=["Measure","Value"]).style.hide(axis="index"))

Measure,Value
Eligible purchase lines,"527,761"
Eligible purchase invoices,"19,773"
Identified customers,"4,334"
Gross purchase value (GBP),"10,271,433.063"
Gross average order value (GBP),519.47


### Leading products by gross purchase value

Product 23843 has a single eligible purchase line of 80,995 units. Separate credit entries exist in the source ledger, so its high gross ranking must not be interpreted as retained spending or widespread demand. Product descriptions below are the latest observed eligible labels; the item code is the analytical identifier.

In [3]:
products=pd.DataFrame(report["leading_products"])
products["Gross value (GBP)"]=products.gross_purchase_millipounds.map(gbp)
display(products.rename(columns={"stock_code_key":"Item code","product_description":"Latest description",
    "purchase_orders":"Invoices","units":"Units"})[["Item code","Latest description","Invoices","Units","Gross value (GBP)"]].style.format({"Invoices":"{:,.0f}","Units":"{:,.0f}"}).hide(axis="index"))

Item code,Latest description,Invoices,Units,Gross value (GBP)
22423,REGENCY CAKESTAND 3 TIER,"1,988","13,879","174,484.740"
23843,"PAPER CRAFT , LITTLE BIRDIE",1,"80,995","168,469.600"
85123A,CREAM HANGING HEART T-LIGHT HOLDER,"2,265","37,952","106,471.280"
47566,PARTY BUNTING,"1,685","18,295","99,504.330"
85099B,JUMBO BAG RED RETROSPOT,"2,089","48,474","94,340.050"
23166,MEDIUM CERAMIC TOP STORAGE JAR,247,"78,033","81,700.920"
23084,RABBIT NIGHT LIGHT,994,"30,788","66,964.990"
22086,PAPER CHAIN KIT 50'S CHRISTMAS,"1,160","19,355","64,952.290"
84879,ASSORTED COLOUR BIRD ORNAMENT,"1,455","36,461","59,094.930"
79321,CHILLI LIGHTS,661,"10,306","54,117.760"


### Country performance

These are source customer-country labels, not independently verified shipping destinations. A country with few customers and large orders can have a high average order value; this does not establish a superior market opportunity.

In [4]:
countries=pd.DataFrame(report["leading_countries"])
countries["Gross value (GBP)"]=countries.gross_purchase_millipounds.map(gbp)
display(countries.rename(columns={"country":"Country","purchase_orders":"Invoices","value_share_percent":"Share (%)"})[
    ["Country","Invoices","Gross value (GBP)","Share (%)"]].style.format({"Invoices":"{:,.0f}","Share (%)":"{:.2f}"}).hide(axis="index"))

Country,Invoices,Gross value (GBP),Share (%)
United Kingdom,"17,901","8,747,561.663",85.16
Netherlands,93,"283,889.340",2.76
EIRE,282,"271,164.300",2.64
Germany,443,"205,569.890",2.00
France,382,"184,582.740",1.80
Australia,56,"138,171.310",1.35
Spain,88,"55,725.110",0.54
Switzerland,50,"53,087.900",0.52
Japan,19,"37,416.370",0.36
Belgium,98,"36,927.340",0.36


### Monthly sales and order value

The month is determined by each invoice's earliest eligible timestamp. November 2011 is the highest complete month by gross value. December 2011 includes observations only through 9 December and is labelled partial. Blank growth cells indicate unavailable or ineligible comparisons, not zero growth.

In [5]:
monthly=pd.DataFrame(report["monthly_sales"])
monthly["Gross value (GBP)"]=monthly.gross_purchase_millipounds.map(gbp)
monthly["Coverage"]=monthly.is_complete_month.map({1:"Full month",0:"Partial through 9 Dec"})
display(monthly.rename(columns={"month":"Month","purchase_orders":"Invoices","gross_aov_gbp":"AOV (GBP)",
    "full_month_growth_percent":"Growth (%)"})[["Month","Invoices","Gross value (GBP)","AOV (GBP)","Coverage","Growth (%)"]].style.format({
    "Invoices":"{:,.0f}","AOV (GBP)":"{:,.2f}","Growth (%)":"{:.2f}"},na_rep="—").hide(axis="index"))

Month,Invoices,Gross value (GBP),AOV (GBP),Coverage,Growth (%)
2010-12,"1,550","777,931.770",501.89,Full month,—
2011-01,"1,081","671,933.190",621.58,Full month,-13.63
2011-02,"1,093","508,886.200",465.59,Full month,-24.27
2011-03,"1,440","691,257.610",480.04,Full month,35.84
2011-04,"1,235","516,218.121",417.99,Full month,-25.32
2011-05,"1,668","741,192.410",444.36,Full month,43.58
2011-06,"1,525","738,751.870",484.43,Full month,-0.33
2011-07,"1,452","689,364.411",474.77,Full month,-6.69
2011-08,"1,339","725,513.500",541.83,Full month,5.24
2011-09,"1,818","1,030,475.361",566.82,Full month,42.03


### Identified-customer purchasing

The denominator is all identified customers with at least one eligible purchase. Missing customer IDs are not pooled. The complete customer-level query output stays in the untracked local processed-data folder.

In [6]:
c=report["customer_purchase_summary"]
display(pd.DataFrame([
    ("Identified customers",f"{c['identified_customers']:,}"),
    ("One eligible invoice",f"{c['one_order_customers']:,}"),
    ("At least two eligible invoices",f"{c['repeat_customers']:,}"),
    ("Repeat-purchase share",f"{c['repeat_customer_share_percent']:.2f}%"),
],columns=["Measure","Value"]).style.hide(axis="index"))

Measure,Value
Identified customers,"4,334"
One eligible invoice,"1,505"
At least two eligible invoices,"2,829"
Repeat-purchase share,65.27%


### Reconciliation evidence

Every row and measure of the six query outputs is compared with pandas. Product, country, month, and order values all sum to the same sales total. Customer values reconcile to identified purchases only. Monetary totals are compared in integer units; ratio checks use a GBP/percentage tolerance of 1e-9.

In [7]:
display(pd.DataFrame(report["reconciliation"].items(),columns=["SQL result","Reconciliation"]).style.hide(axis="index"))
assert all(value=="all_rows_and_measures_passed" for value in report["reconciliation"].values())
assert pd.isna(monthly.loc[monthly.month.eq("2011-12"),"full_month_growth_percent"]).all()
print("Full query outputs: data/processed/sql_analysis/. Bounded analytical summary: reports/04_sql_summary.json.")

SQL result,Reconciliation
sales_summary,all_rows_and_measures_passed
products,all_rows_and_measures_passed
countries,all_rows_and_measures_passed
monthly_sales,all_rows_and_measures_passed
orders,all_rows_and_measures_passed
customers,all_rows_and_measures_passed


Full query outputs: data/processed/sql_analysis/. Bounded analytical summary: reports/04_sql_summary.json.


## Takeaways

- SQL and pandas agree on the reviewed purchase populations and all six analytical outputs.
- Gross purchase value and order count describe different grains; average order value uses invoices.
- Product rankings require transaction context: a single extreme purchase can dominate one product's value.
- The UK accounts for 85.16% of gross value, while customer analysis covers identified purchases only.
- The next analytical stage will explore distributions and trends with figures. Credit matching, causal explanations, and RFM scoring have not been performed here.